# WeaveGuard AI — Étape 07 : diagnostic du biais de domaine (classe ↔ tissu)
**Constat** : avec 6 statistiques de couleur seules, le plus proche voisin partage une classe de défaut dans 54 % des cas (hasard : 12,6 %).

Ce notebook **ne modifie aucun modèle** et ne règle aucun seuil. Il mesure :
1. le lien entre l'apparence du tissu (clusters) et la classe de défaut (NMI, comparé à un hasard par permutation) ;
2. la couverture des partitions val/test par les groupes d'apparence présents en train ;
3. les performances de YOLO selon la **familiarité** de chaque image avec le train (rappel avec intervalle de confiance par bootstrap).

Analyse **descriptive et post hoc** sur val + test : elle ne sert pas à choisir un modèle, donc le test n'est pas ré-utilisé pour une sélection.

In [ ]:
# 1) Drive + dépendances
from google.colab import drive
drive.mount('/content/drive')
!pip -q install ultralytics
import torch
print('GPU :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'AUCUN (plus lent)')

In [ ]:
# 2) Configuration : on relit le modèle YOLO exporté (poids, seuil de confiance, taille d'entrée)
from pathlib import Path
import json, hashlib, zipfile, random, collections
import numpy as np, pandas as pd
from PIL import Image, ImageOps
import torch, torch.nn as nn, torch.nn.functional as F, torchvision, torchvision.transforms as T
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm
from ultralytics import YOLO

WG = Path('/content/drive/MyDrive/WeaveGuard'); DRIVE_OUT = WG / 'datasets'
V2 = Path('/content/weaveguard_dataset_v2'); SEED = 42
YOLO_DIR = WG / 'models' / 'weaveguard_yolo_v1'
card = json.load(open(YOLO_DIR / 'model_card.json'))
CONF, IMGSZ = card['conf_threshold']['value'], card['imgsz']
print('Modèle YOLO :', card['selected_experiment'], '| imgsz', IMGSZ, '| conf', round(CONF, 3))
dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# 3) Utilitaires sans PyTorch : appariement, bootstrap, NMI
import numpy as np
from sklearn.metrics import normalized_mutual_info_score

def iou_xyxy(a, b):
    w = min(a[2], b[2]) - max(a[0], b[0]); h = min(a[3], b[3]) - max(a[1], b[1])
    if w <= 0 or h <= 0: return 0.0
    i = w * h; u = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - i
    return i / u if u > 0 else 0.0

def match_image(pred, gt, thr=0.5):
    # pred : [(cls, conf, x0, y0, x1, y1)] ; gt : [(cls, x0, y0, x1, y1)]
    # retourne : GT retrouvées (classe correcte), GT retrouvées (toute classe), nombre de prédictions non appariées
    pred = sorted(pred, key=lambda p: -p[1]); used = set(); tp = 0
    for p in pred:
        best, bj = 0.0, -1
        for j, g in enumerate(gt):
            if j in used or g[0] != p[0]: continue
            v = iou_xyxy(p[2:], g[1:])
            if v >= thr and v > best: best, bj = v, j
        if bj >= 0: used.add(bj); tp += 1
    found_cls = [j in used for j in range(len(gt))]
    found_any = [any(iou_xyxy(p[2:], g[1:]) >= thr for p in pred) for g in gt]
    return found_cls, found_any, len(pred) - tp

def boot_ratio(found, total, n=2000, seed=0):
    found, total = np.asarray(found, float), np.asarray(total, float)
    if total.sum() == 0: return (float('nan'),) * 3
    rng = np.random.RandomState(seed); idx = rng.randint(0, len(found), (n, len(found)))
    r = found[idx].sum(1) / np.maximum(total[idx].sum(1), 1)
    return float(found.sum() / total.sum()), float(np.percentile(r, 2.5)), float(np.percentile(r, 97.5))

def nmi_with_null(y, labels, n_perm=200, seed=0):
    rng = np.random.RandomState(seed); real = normalized_mutual_info_score(y, labels)
    null = [normalized_mutual_info_score(rng.permutation(y), labels) for _ in range(n_perm)]
    return float(real), float(np.mean(null)), float(np.percentile(null, 95))

In [ ]:
# 4) Dataset v2 en local (intégrité) + embeddings ResNet-50 de TOUTES les images (même extracteur que la similarité)
if not V2.exists(): zipfile.ZipFile(DRIVE_OUT / 'weaveguard_dataset_v2.zip').extractall(V2)
stats = json.load(open(DRIVE_OUT / 'dataset_stats_v2.json')); man = pd.read_csv(V2 / 'manifest.csv')
h = hashlib.sha256('\n'.join(sorted(f'{a}|{b}|{c}' for a, b, c in zip(man.name, man.md5, man.split))).encode()).hexdigest()
assert h == stats['dataset_hash_sha256'], 'Le dataset local ne correspond pas au v2 de référence !'
NAMES = stats['classes']
man['primary'] = man.classes.map(lambda c: str(c).split(';')[0])      # classe principale (41 images sont multi-classes)

class Embedder(nn.Module):
    def __init__(self):
        super().__init__(); m = torchvision.models.resnet50(weights='IMAGENET1K_V1')
        self.body = nn.Sequential(m.conv1, m.bn1, m.relu, m.maxpool, m.layer1, m.layer2, m.layer3, m.layer4)
    @torch.no_grad()
    def forward(self, x): return F.normalize(self.body(x).mean(dim=(2, 3)), dim=1)
emb = Embedder().to(dev).eval()
prep = T.Compose([T.Resize(256), T.CenterCrop(256), T.ToTensor(), T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
paths = [V2 / 'images' / s / n for s, n in zip(man.split, man.name)]
def lp(p): return prep(Image.open(p).convert('RGB'))
feats = []
with ThreadPoolExecutor(8) as ex:
    for i in tqdm(range(0, len(paths), 32), desc='embeddings'):
        feats.append(emb(torch.stack(list(ex.map(lp, paths[i:i + 32]))).to(dev)).cpu().numpy())
X = np.concatenate(feats).astype('float32')

def color_feat(p):
    a = np.asarray(Image.open(p).convert('RGB').resize((64, 64)), dtype=np.float32) / 255
    return np.r_[a.mean((0, 1)), a.std((0, 1))]
C = np.stack([color_feat(p) for p in paths]); C = (C - C.mean(0)) / C.std(0)
print(X.shape, C.shape)

In [ ]:
# 5) Lien apparence du tissu <-> classe de défaut (NMI vs hasard par permutation)
from sklearn.cluster import KMeans
y = man.primary.values; rows = []
for k in (5, 10, 20):
    for name, Z in (('ResNet-50', X), ('couleur (6 nombres)', C)):
        lab = KMeans(k, n_init=10, random_state=SEED).fit_predict(Z); real, nm, n95 = nmi_with_null(y, lab)
        rows.append(dict(k=k, descripteur=name, NMI=round(real, 3), NMI_hasard_moy=round(nm, 3), NMI_hasard_p95=round(n95, 3)))
nmi_tbl = pd.DataFrame(rows); display(nmi_tbl)
print('NMI : 0 = aucun lien entre groupe d apparence et classe ; 1 = la classe est déterminée par le groupe.')

K = 10; lab10 = KMeans(K, n_init=10, random_state=SEED).fit_predict(X); man['cluster'] = lab10
print('\nClusters (k=10, ResNet) x classe principale :'); display(pd.crosstab(man.cluster, man.primary))
print('Clusters x partition :'); ct_split = pd.crosstab(man.cluster, man.split)[['train', 'val', 'test']]; display(ct_split)
top_share = {c: round(float(man[man.primary == c].cluster.value_counts(normalize=True).iloc[0]), 2) for c in sorted(man.primary.unique())}
print('Part de chaque classe dans son cluster principal :', top_share)
train_clusters = ct_split.index[ct_split.train >= 5]
vt = man[man.split.isin(['val', 'test'])]
print(f"Images val/test dont le cluster compte >= 5 images de train : {vt.cluster.isin(train_clusters).mean():.1%}")

In [ ]:
# 6) Robustesse de YOLO selon la familiarité avec le train (analyse descriptive, aucun réglage)
tr = np.where(man.split == 'train')[0]; ev = np.where(man.split.isin(['val', 'test']))[0]
fam = (X[ev] @ X[tr].T).max(axis=1)                       # similarité cosinus max avec une image de train
q1, q2 = np.quantile(fam, [1 / 3, 2 / 3])
tier = np.where(fam <= q1, '1_peu familier', np.where(fam <= q2, '2_intermédiaire', '3_familier'))
print(f'Familiarité : min {fam.min():.3f} | tiers {q1:.3f} / {q2:.3f} | max {fam.max():.3f}')

model = YOLO(str(YOLO_DIR / 'best.pt'))
def read_gt(split, p, W, H):
    out = []
    for line in (V2 / 'labels' / split / (Path(p).stem + '.txt')).read_text().splitlines():
        s = line.split()
        if len(s) == 5:
            c = int(s[0]); cx, cy, bw, bh = map(float, s[1:]); out.append((c, (cx-bw/2)*W, (cy-bh/2)*H, (cx+bw/2)*W, (cy+bh/2)*H))
    return out

per_img = []
ev_paths = [paths[i] for i in ev]
for k, r in enumerate(tqdm(model.predict(source=[str(p) for p in ev_paths], conf=CONF, imgsz=IMGSZ, iou=0.6, stream=True, verbose=False), total=len(ev_paths), desc='YOLO')):
    i = ev[k]; H_, W_ = r.orig_shape
    b = r.boxes; pred = [(int(c), float(s), *map(float, xy)) for c, s, xy in zip(b.cls.cpu().numpy(), b.conf.cpu().numpy(), b.xyxy.cpu().numpy())]
    gt = read_gt(man.split[i], man.name[i], W_, H_)
    fc, fa, fp = match_image(pred, gt)
    per_img.append(dict(i=int(i), split=man.split[i], tier=tier[k], fam=float(fam[k]), n_gt=len(gt), found_cls=int(sum(fc)), found_any=int(sum(fa)), fp=int(fp),
                        gt_classes=[NAMES[g[0]] for g in gt], gt_found=[bool(x) for x in fc]))
pi = pd.DataFrame(per_img)

rows = []
for t in sorted(pi.tier.unique()):
    d = pi[pi.tier == t]
    rc = boot_ratio(d.found_cls, d.n_gt); ra = boot_ratio(d.found_any, d.n_gt)
    rows.append(dict(tier=t, images=len(d), boîtes_GT=int(d.n_gt.sum()), rappel_classe=round(rc[0], 3), IC95_classe=f'[{rc[1]:.2f}, {rc[2]:.2f}]',
                     rappel_localisation=round(ra[0], 3), IC95_loc=f'[{ra[1]:.2f}, {ra[2]:.2f}]', FP_par_image=round(float(d.fp.mean()), 2)))
tier_tbl = pd.DataFrame(rows); display(tier_tbl)

from scipy.stats import spearmanr
flat = [(r.fam, f) for r in pi.itertuples() for f in r.gt_found]
rho, pv = spearmanr([a for a, _ in flat], [int(b) for _, b in flat])
print(f'Corrélation de Spearman familiarité <-> boîte retrouvée : rho={rho:.3f}, p={pv:.3g} ({len(flat)} boîtes)')
print('Mix de classes par tiers (images contenant la classe) :'); display(pd.crosstab(pd.Series(tier, name='tier'), man.primary.values[ev]))
print('NB : les tiers n ont pas le même mix de classes ; les IC sont larges (petits effectifs). À lire comme un indice, pas comme une preuve.')

In [ ]:
# 7) Rapport à coller dans le chat
rep = dict(dataset_hash=h[:16], yolo=card['selected_experiment'], conf=CONF, imgsz=IMGSZ,
           nmi=nmi_tbl.to_dict('records'), class_top_cluster_share=top_share,
           valtest_in_trained_clusters=round(float(vt.cluster.isin(train_clusters).mean()), 3),
           familiarity_quantiles=[float(fam.min()), float(q1), float(q2), float(fam.max())],
           tiers=tier_tbl.to_dict('records'), spearman=dict(rho=float(rho), p=float(pv), n_boxes=len(flat)),
           note='analyse descriptive post hoc sur val+test ; aucun modèle ni seuil modifié')
(WG / 'models' / 'domain_bias_report_v1.json').write_text(json.dumps(rep, indent=1, ensure_ascii=False, default=str), encoding='utf-8')
print(json.dumps(rep, indent=1, ensure_ascii=False, default=str))